In [ ]:
# IMPORTANT!!! Change from CPU into T4 GPU

!nvidia-smi

Sat Apr 25 23:30:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Installation of PyCUDA package to work in any device withouth NVIDIA

!pip install pycuda

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 59.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 14.3 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2026.1-cp312-cp312-linux_x86_64.whl size=659447 sha256=1a2ec64745086bf4119d0645ebe92e4ea6f786552f48d8e1e85c4c18cab3a68a
  Stored in directory: /root/.cache/pip/wheels/90/2a/71/75ec0cc316cc0ff494bfffa2935e02580129cb7f859a0cfd8f
Successfully built pycuda


In [ ]:
import pycuda.autoinit
import pycuda.driver as cuda
import numpy as np
from pycuda.compiler import SourceModule

In [ ]:
#first, image generation as an array to focus in the CUDA application

img = np.random.randint(0, 256, size=1024).astype(np.uint8)
n = img.size

In [ ]:
#memory on GPU
img_gpu = cuda.mem_alloc(img.nbytes)
out_gpu = cuda.mem_alloc(img.nbytes)

#copy data
cuda.memcpy_htod(img_gpu, img)

#CUDA kernel
kernel_code = """
__global__ void negativo(unsigned char *img, unsigned char *out, int n)
{
    int i = threadIdx.x + blockIdx.x * blockDim.x;

    if (i < n)
    {
        out[i] = 255 - img[i];
    }
}
"""
mod = SourceModule(kernel_code)
negativo = mod.get_function("negativo")

#executing conf
block_size = 256
grid_size = (n + block_size - 1) // block_size

#kernel exe
negativo(img_gpu, out_gpu, np.int32(n),
         block=(block_size,1,1),
         grid=(grid_size,1))

#recover result
out = np.empty_like(img)
cuda.memcpy_dtoh(out, out_gpu)

#result verification
print("Original:", img[:10])
print("Negativo:", out[:10])

Original: [211  50 195 248 141 165  24   7  79 212]
Negativo: [ 44 205  60   7 114  90 231 248 176  43]


In [ ]:
np.all(out == 255 - img)

#to confirm the result was adecquate,
#using the equation provided on the documentation

np.True_